# NeurIPS 2026 Smart Buildings — Version 14: Constrained QP Dispatch

**Independent classical-control experiment:** Use convex constrained optimization, rather than PID, model identification, forecasting, or finite-mode MPC, to distribute limited HVAC actuation among zones.

## Decision architecture

1. Decode normalized zone/outside-air temperatures using per-building metadata, and derive per-zone heating/cooling violations from live setpoints.
2. Compute a physics-informed AHU supply-air setpoint. This preserves the known fact that an AHU **only cools** mixed air; heating requires the separate VAV reheat coils.
3. **AHU airflow QP:** For each AHU, minimize a convex sum of squared deviation from each zone's requested damper opening plus a small energy regularizer; constrain each damper to its physical opening range and total **incremental** damper opening by an AHU sharing budget.
4. **Shared hot-water QP:** Across cold zones, minimize a similar quadratic objective in `damper × reheat_valve` units, with upper bounds set by allocated damper and a building-wide capacity proxy.
5. Recover normalized actuator commands, switch on only needed equipment, and return the exact `float32`/`int32` Kaggle action contract.

## Optimization

For both QPs, solve `min 0.5 Σ wᵢ(xᵢ-dᵢ)² + 0.5 λ_reg Σ xᵢ²` subject to `0 ≤ xᵢ ≤ uᵢ` and `Σ xᵢ ≤ B`. The comfort weights `wᵢ` increase with temperature violations, including an extra penalty for excursions beyond 5 K. The dual has a monotonic scalar shadow price: `xᵢ(μ) = clip((wᵢdᵢ−μ)/(wᵢ+λ_reg), 0, uᵢ)`. Fixed-iteration bisection solves it without SciPy or model inference.

**Important limitation:** The budgets constrain normalized damper/reheat *command proxies*, **not calibrated airflow in m³/s or thermal power in kW**. They are chosen to make competition-safe constrained resource sharing tractable without hidden plant coefficients. This is a legitimate convex QP, not an exact physical optimization model. The numeric constants are exploratory and need ablation on public training scenarios.

## Computational cost and validation

One AHU QP per active AHU, plus one hot-water QP if heating is needed; 34 dual bisection iterations per binding problem. Complexity is O(34 × number of zones) per step, without any planner tree or online identification, and with bounded memory.

Run `python Co_Kaggle/g8/submissions/test_version14.py` locally for synthetic interface, convex-solver, safety, and micro-benchmark checks. Full official simulator and Kaggle scores have **not** yet been measured. Compare to Version 7 (`0.03947` public score), not to the invalid all-zero reference.


In [ ]:
"""Version 14: convex QP dispatch for shared HVAC resources.

Each 5-minute decision solves two bounded quadratic programs: AHU airflow
allocation and building-wide VAV reheat allocation. Plant capacity is expressed
as an explicit *normalized command proxy*, not a calibrated CFM/kW guarantee.
A monotone water-filling dual solve needs no SciPy, online learning or MPC.
"""
import re

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


def allocate_qp(desired, priority, upper, capacity, regularizer=0.08):
    """Solve a separable convex QP with a shared linear capacity constraint.

    min_x 0.5 * sum_i priority[i] * (x[i] - desired[i])**2
                  + 0.5 * regularizer * sum_i x[i]**2
    s.t. 0 <= x[i] <= upper[i]; sum_i x[i] <= capacity.

    KKT gives x_i(lambda) = clip((priority_i*desired_i-lambda) /
                                    (priority_i+regularizer), 0, upper_i).
    Bisection finds the nonnegative shadow price lambda for the resource cap.
    """
    desired = np.asarray(desired, dtype=np.float64)
    priority = np.asarray(priority, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)
    if desired.ndim != 1 or priority.shape != desired.shape or upper.shape != desired.shape:
        raise ValueError('QP inputs must have matching one-dimensional shapes')
    if not (np.isfinite(desired).all() and np.isfinite(priority).all()
            and np.isfinite(upper).all() and np.isfinite(capacity)):
        raise ValueError('QP input contains non-finite values')
    if np.any(priority <= 0.0) or np.any(upper < 0.0) or capacity < 0.0:
        raise ValueError('QP requires positive priorities and nonnegative limits')
    if desired.size == 0 or capacity == 0.0:
        return np.zeros_like(desired)

    linear = priority * np.maximum(desired, 0.0)
    curvature = priority + regularizer
    optimum = np.minimum(upper, linear / curvature)
    if float(optimum.sum()) <= capacity:
        return optimum

    low, high = 0.0, float(np.max(linear))
    for _ in range(34):
        price = 0.5 * (low + high)
        proposal = np.clip((linear - price) / curvature, 0.0, upper)
        if float(proposal.sum()) > capacity:
            low = price
        else:
            high = price
    return np.clip((linear - high) / curvature, 0.0, upper)


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Metadata-indexed hybrid action controller with explicit QP constraints."""

    ACTIVE_K = 0.18
    MIN_DAMPER = 0.10

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.continuous_count = int(meta['n_continuous_action'])
        self.discrete_count = int(meta['n_discrete_action'])
        self.fields = {str(name): i for i, name in enumerate(meta['observation_field_names'])}
        self.temp_actions = [int(i) for i in meta['temp_indices']]
        self.pressure_actions = [int(i) for i in meta['pressure_indices']]
        self.recirc_actions = [int(i) for i in meta['recirculation_indices']]
        self.water_actions = [int(i) for i in meta['water_indices']]
        self.water_temperature = self.water_actions[0] if self.water_actions else None
        self.water_pressure = (self.pressure_actions[len(self.temp_actions)]
                               if len(self.pressure_actions) > len(self.temp_actions) else None)
        self.water_switch = self.discrete_count - 1
        self.comfort = tuple(map(float, meta['comfort_temp_window']))
        self.eco = tuple(map(float, meta['eco_temp_window']))
        self.morning = float(meta['morning_start_hour'])
        self.evening = float(meta['evening_start_hour'])

        normalizers = meta.get('observation_normalization_metadata', {})
        zone = normalizers.get('zone_air_temperature_sensor', {})
        self.zone_mean = float(zone.get('sample_mean', np.nan))
        self.zone_std = float(zone.get('sample_std', np.nan))
        outside = normalizers.get('outside_air_temperature_sensor', {})
        self.outside_mean = float(outside.get('sample_mean', np.nan))
        self.outside_std = float(outside.get('sample_std', np.nan))
        if not (np.isfinite(self.zone_mean) and np.isfinite(self.zone_std)
                and self.zone_std > 0.0):
            raise ValueError('Zone temperature normalization is required')
        if not (np.isfinite(self.outside_mean) and np.isfinite(self.outside_std)
                and self.outside_std > 0.0):
            self.outside_mean = self.outside_std = np.nan

        def numeric_order(key):
            match = re.search(r'\d+', str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta['ahu_zone_assignments']
        vav_actions = meta['zone_action_indices']
        for i, ahu_name in enumerate(sorted(assignments, key=numeric_order)[:len(self.temp_actions)]):
            zones = []
            for room in assignments[ahu_name]:
                vav_name = str(room).replace('room_', 'VAV_')
                sensor = self.fields.get(vav_name + '_zone_air_temperature_sensor')
                if sensor is None:
                    continue
                actuators = vav_actions.get(vav_name, {})
                zones.append({
                    'temperature': sensor,
                    'low': self.fields.get(vav_name + '_zone_air_heating_temperature_setpoint_sensor'),
                    'high': self.fields.get(vav_name + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper': actuators.get('supply_air_damper_percentage_command'),
                    'reheat': actuators.get('reheat_valve_percentage_command'),
                })
            self.ahus.append({
                'zones': zones, 'switch': i, 'supply': self.temp_actions[i],
                'pressure': self.pressure_actions[i] if i < len(self.pressure_actions) else None,
                'recirc': self.recirc_actions[i] if i < len(self.recirc_actions) else None,
                'outdoor': self.fields.get('ahs_' + str(ahu_name) + '_outside_air_temperature_sensor'),
            })
        self.reset()

    def reset(self):
        """Stateless controller: no stale episode data to clear."""
        return None

    @staticmethod
    def _command(fraction):
        return float(np.clip(2.0 * fraction - 1.0, -1.0, 1.0))

    @staticmethod
    def _read(obs, idx):
        if idx is None or not isinstance(idx, (int, np.integer)) or not 0 <= idx < len(obs):
            return np.nan
        return float(obs[idx])

    def _fallback_band(self, obs):
        def cyclic(sin_name, cos_name, period):
            sine = self._read(obs, self.fields.get(sin_name))
            cosine = self._read(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2.0*np.pi) * period/(2.0*np.pi))
        hour = cyclic('hod_sin_000', 'hod_cos_000', 24.0)
        day = cyclic('dow_sin_000', 'dow_cos_000', 7.0)
        if hour is not None and day is not None and day < 5 and self.morning <= hour < self.evening:
            return self.comfort
        return self.eco

    def _zone_states(self, obs, ahu, fallback):
        outdoors = self._read(obs, ahu['outdoor'])
        outdoor = (self.outside_mean + self.outside_std * outdoors
                   if np.isfinite(self.outside_std) and np.isfinite(outdoors) else np.nan)
        if not np.isfinite(outdoor) or not 180.0 <= outdoor <= 350.0:
            outdoor = np.nan
        states = []
        for zone in ahu['zones']:
            raw = self._read(obs, zone['temperature'])
            temperature = self.zone_mean + self.zone_std * raw
            if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
                continue
            low, high = self._read(obs, zone['low']), self._read(obs, zone['high'])
            if not (np.isfinite(low) and np.isfinite(high)
                    and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
                low, high = fallback
            states.append({'zone': zone, 'temp': temperature,
                           'hot': max(0.0, temperature - high),
                           'cold': max(0.0, low - temperature)})
        return states, outdoor

    @staticmethod
    def _priority(violation):
        """Raise QP marginal regret for the >5K comfort-penalty tail."""
        return 1.0 + 1.8 * min(violation, 5.0) + 2.0 * max(0.0, violation - 5.0)**2

    def _supply(self, states, outdoor):
        """Physics-informed AHU operating point; QP allocates its VAV output."""
        max_hot = max(s['hot'] for s in states)
        return_air = float(np.mean([s['temp'] for s in states]))
        if max_hot < self.ACTIVE_K:
            supply = 305.0
        else:
            hottest = [s['temp'] for s in states if s['hot'] > 0.0]
            supply = float(np.clip(np.mean(hottest) - 4.0 - 1.3*max_hot, 285.0, 297.0))
        recirc = 0.70
        if (max_hot >= self.ACTIVE_K and np.isfinite(outdoor)
                and 287.0 <= outdoor < return_air - 1.5):
            mixed_min = 0.15*return_air + 0.85*outdoor
            if mixed_min <= supply:
                recirc = float(np.clip((supply - outdoor)/(return_air - outdoor), 0.15, 0.70))
        mixed = recirc*return_air + (1.0 - recirc)*(outdoor if np.isfinite(outdoor) else return_air)
        delivered = min(mixed, supply)
        return supply, recirc, delivered

    def _dispatch_ahu(self, ahu, states, outdoor, continuous, discrete):
        """Solve AHU damper QP; return candidate reheat loads for global QP."""
        if not states:
            return []
        active = any(s['hot'] >= self.ACTIVE_K or
                     (s['cold'] >= self.ACTIVE_K and s['zone']['reheat'] is not None)
                     for s in states)
        if not active:
            return []
        supply, recirc, delivered = self._supply(states, outdoor)
        desired, weights, upper = [], [], []
        for state in states:
            hot, cold, zone = state['hot'], state['cold'], state['zone']
            if hot >= self.ACTIVE_K:
                headroom = max(2.5, state['temp'] - delivered)
                damper = float(np.clip(0.12 + 2.1*hot/headroom, 0.10, 0.98))
            elif cold >= self.ACTIVE_K and zone['reheat'] is not None:
                damper = float(np.clip(0.22 + 0.21*cold, 0.18, 0.90))
            else:
                damper = self.MIN_DAMPER
            # A missing mapped damper cannot be allocated additional airflow.
            available = 0.88 if zone['damper'] is not None else 0.0
            desired.append(max(0.0, damper-self.MIN_DAMPER))
            weights.append(self._priority(max(hot, cold)))
            upper.append(available)
        # Up to 0.98 open for the first zone; shared AHU proxy becomes tight
        # when multiple VAVs simultaneously demand nearly full opening.
        capacity = 0.88 + 0.61 * (len(states) - 1)
        extra = allocate_qp(desired, weights, upper, capacity)
        dampers = self.MIN_DAMPER + extra
        discrete[ahu['switch']] = 1
        continuous[ahu['supply']] = float(np.clip((supply - 295.0)/10.0, -1.0, 1.0))
        if ahu['recirc'] is not None:
            continuous[ahu['recirc']] = self._command(recirc)
        if ahu['pressure'] is not None:
            loading = max(float(dampers.mean()), 0.5*float(dampers.max()))
            pressure_pa = 200.0 + 1250.0*np.sqrt(np.clip((loading-0.10)/0.65, 0.0, 1.0))
            continuous[ahu['pressure']] = float(np.clip(pressure_pa/1000.0 - 1.0, -1.0, 1.0))
        heating = []
        for state, damper in zip(states, dampers):
            zone = state['zone']
            if zone['damper'] is not None:
                continuous[zone['damper']] = self._command(damper)
            if state['cold'] >= self.ACTIVE_K and zone['reheat'] is not None:
                heating.append((state, float(damper)))
        return heating

    def _dispatch_reheat(self, loads, outdoors, continuous, discrete):
        """Shared-water QP in delivered damper*valve units, then invert valve."""
        if not loads:
            return
        desired, weights, upper = [], [], []
        for state, damper in loads:
            target_valve = float(np.clip(0.16 + 0.31*state['cold'], 0.0, 1.0))
            desired.append(damper*target_valve)
            weights.append(self._priority(state['cold']))
            upper.append(damper)
        capacity = 0.85 + 0.60 * (len(loads) - 1)
        allocation = allocate_qp(desired, weights, upper, capacity)
        if not np.any(allocation > 1e-6):
            return
        discrete[self.water_switch] = 1
        for (state, damper), heat in zip(loads, allocation):
            continuous[state['zone']['reheat']] = self._command(float(heat/damper))
        worst = max(state['cold'] for state, _ in loads)
        outside = float(np.mean(outdoors)) if outdoors else 294.0
        water_k = float(np.clip(318.0 + 0.35*max(0.0, 294.0-outside) + 5.0*worst,
                                320.0, 349.0))
        if self.water_temperature is not None:
            continuous[self.water_temperature] = self._command((water_k-310.0)/40.0)
        if self.water_pressure is not None:
            mean_valve = float(np.mean([heat/d for heat, (_, d) in zip(allocation, loads)]))
            continuous[self.water_pressure] = self._command(float(np.clip(0.38+0.52*mean_valve, 0.38, 0.95)))

    def get_action(self, obs):
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous = np.full(self.continuous_count, -1.0, dtype=np.float32)
        discrete = np.zeros(self.discrete_count, dtype=np.int32)
        for index in self.temp_actions + self.recirc_actions:
            continuous[index] = 1.0
        fallback = self._fallback_band(obs)
        heating, outdoors = [], []
        for ahu in self.ahus:
            states, outside = self._zone_states(obs, ahu, fallback)
            if np.isfinite(outside):
                outdoors.append(outside)
            heating.extend(self._dispatch_ahu(ahu, states, outside, continuous, discrete))
        self._dispatch_reheat(heating, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
